# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
1. No spaghetti — multiple lines must use grey + single highlight
2. Remove clutter: no chart borders, no heavy gridlines, no legend if you can label directly
3. Insight title — states the finding, not the topic
4. Carry forward from Lecture 2: white background, Arial font, professional quality


In [3]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('/Users/sujith/Documents/Data_Visualization/data-viz-class-material/data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [4]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task 1 — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#DDDDDD), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

> 💡 `df[df['Region'] == 'Asia']` to filter; use `go.Figure()` with a loop for per-country control


In [5]:
# Task 1 — Multi-series line with highlight
# YOUR CODE HERE

asia_df = df[df['Region'] == 'Asia']

highlight_country = 'China'   # you can change this (India also works well)

fig = go.Figure()

# Add all countries (grey lines)
for country in asia_df['Country'].unique():
    country_df = asia_df[asia_df['Country'] == country]
    
    if country == highlight_country:
        fig.add_trace(go.Scatter(
            x=country_df['Year'],
            y=country_df['CO2_Mt'],
            mode='lines',
            line=dict(color='#E74C3C', width=3),
            name=country,
            showlegend=False
        ))
        
        # Direct label at end
        fig.add_annotation(
            x=country_df['Year'].max(),
            y=country_df['CO2_Mt'].iloc[-1],
            text=country,
            showarrow=False,
            xanchor='left',
            font=dict(size=12, color='#E74C3C')
        )
        
    else:
        fig.add_trace(go.Scatter(
            x=country_df['Year'],
            y=country_df['CO2_Mt'],
            mode='lines',
            line=dict(color='#DDDDDD', width=1),
            showlegend=False
        ))

# Styling
fig.update_layout(
    title="China’s emissions surge far beyond other Asian countries, dominating regional CO2 growth since 2000",
    xaxis_title="Year",
    yaxis_title="CO2 Emissions (Mt)",
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family="Arial", size=12)
)

# Light grid only
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='lightgray')

fig.show()

---
## Task 2 — Slopegraph: Regional Change 2000 vs 2022

**What to build:** A slopegraph comparing **average regional CO2 emissions** between 2000 and 2022.

**Requirements:**
- One line per region (not per country — aggregate first)
- Colour: regions that increased = one colour; decreased = another
- Values labelled at both ends of each line
- No y-axis tick labels (the endpoint labels make them redundant)
- Insight title stating which regions moved most

> 💡 `df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()` then filter to 2000 and 2022


In [9]:
# Task 2 — Slopegraph: regional averages
# YOUR CODE HERE

# Step 1: Aggregate
region_year = (df.groupby(['Region', 'Year'])['CO2_Mt']
               .mean()
               .reset_index())

# Step 2: Filter years
slope_df = region_year[region_year['Year'].isin([2000, 2022])]

# Step 3: Pivot
pivot_df = slope_df.pivot(index='Region', columns='Year', values='CO2_Mt').reset_index()

fig = go.Figure()

for _, row in pivot_df.iterrows():
    start = row[2000]
    end = row[2022]
    
    # Better color contrast
    color = '#1f77b4' if end > start else '#ff7f0e'   # blue vs orange
    
    fig.add_trace(go.Scatter(
        x=[2000, 2022],
        y=[start, end],
        mode='lines+markers',
        line=dict(color=color, width=3),
        marker=dict(size=6),
        showlegend=False
    ))
    
    # Labels
    fig.add_annotation(
        x=2000, y=start,
        text=f"{row['Region']} ({start:.0f})",
        showarrow=False,
        xanchor='right',
        font=dict(size=10)
    )
    
    fig.add_annotation(
        x=2022, y=end,
        text=f"{row['Region']} ({end:.0f})",
        showarrow=False,
        xanchor='left',
        font=dict(size=10)
    )

# Layout
fig.update_layout(
    title="Asia shows the steepest rise in emissions since 2000, while few regions show any meaningful decline",
    xaxis=dict(showgrid=False, tickvals=[2000, 2022]),
    yaxis=dict(showgrid=False, showticklabels=False),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family="Arial", size=12)
)

fig.show()